In [20]:
%pip install -q \
  "langchain>=0.3" \
  "langchain-core>=0.3" \
  "langchain-openai>=0.2" \
  "langchain-community>=0.3" \
  "langchain-chroma>=0.2" \
  "openai>=1.50" \
  faiss-cpu chromadb datasets tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6

In [22]:
import os
import pickle
import pandas as pd
import numpy as np
import faiss

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
PROJECT_PATH = "/content/drive/MyDrive/KeepCoding/Ai-Engineering/Practica"

DATA_PATH = os.path.join(PROJECT_PATH, "data")
os.makedirs(DATA_PATH, exist_ok=True)

Cargo los embeddings

In [7]:
embeddings_path =os.path.join(DATA_PATH, "event_embeddings.pkl")
with open(embeddings_path, "rb") as f:
    all_embeddings = pickle.load(f)

In [11]:
print(len(all_embeddings))
print(len(all_embeddings[0]))
print(all_embeddings[0])

2500
1024
[-0.0026226044, 0.013748169, 0.04626465, -0.0005927086, -0.025360107, 0.02154541, -0.012611389, -0.0309906, -0.021148682, -0.034210205, 0.022445679, -0.017150879, -0.017349243, -0.028945923, -0.02709961, -0.020996094, -0.013824463, -0.032196045, -0.041229248, -0.0030269623, -0.0025310516, 0.04751587, 0.01247406, -0.018478394, 0.047058105, -0.026382446, -0.015151978, 0.06402588, 0.02508545, -0.057800293, 0.013748169, 0.0013856888, 0.0013723373, -0.00844574, -0.0065841675, -0.004886627, -0.0023460388, -0.06506348, -0.001868248, 0.0546875, -0.016525269, 0.017028809, 0.014533997, 0.0041389465, -0.006198883, 0.047088623, -0.022216797, -0.05496216, 0.044891357, 0.009109497, 0.014266968, 0.03869629, -0.008590698, -0.013442993, -0.017181396, 0.034210205, 0.015731812, 0.0030326843, 0.015289307, -0.027511597, 0.008079529, -0.0070152283, 0.017913818, 0.018035889, -0.025527954, -0.005908966, -0.010475159, 0.08416748, 0.00687027, 0.029037476, 0.02418518, 0.055267334, 0.01637268, 0.015625,

Cargo ahora mi dataframe

In [15]:
documents_path =os.path.join(DATA_PATH, "event_documents.pkl")
with open(documents_path, "rb") as f:
    documents = pickle.load(f)

In [17]:
print(len(documents))
print(documents[0])

2500

  Evento: Fiesta reggaeton

  Ciudad: Málaga

  Tipo de recinto: Terraza

  Género musical: Techno

  Público objetivo: Erasmus

  Día de la semana: Viernes

  Hora de inicio: 00:00

  Precio de entrada: 12 €

  Aforo: 1200 personas

  Ocupación: 70%

  Inicio de ventas: 7 días antes

  Estrategia de promoción: Promoción grupos

  Canal principal de venta: Fourvenues

  Copas incluidas: 2

  Climatología: Calor

  Eventos competidores: 0

  Resultado del evento: Buen resultado

  Observaciones comerciales: Promoción por grupos funcionó muy bien
  


En clase usan `FAISS.from_documents()` porque trabajan con objetos `Document` de LangChain y generan los embeddings dentro del propio flujo.

En mi caso ya tengo los embeddings calculados y guardados con Cohere (`event_embeddings.pkl`), por lo que utilizaré FAISS directamente para reutilizarlos y evitar volver a generar embeddings.


In [21]:
embeddings_array = np.array(all_embeddings).astype("float32")

print(embeddings_array.shape)

(2500, 1024)


In [24]:
dimension = embeddings_array.shape[1]
# Obtengo la dimensión de los embeddings. En mi caso cada evento está representado por un vector de 1024 posiciones.
index = faiss.IndexFlatL2(dimension)
# Creo un índice FAISS utilizando distancia euclídea (L2), que servirá para buscar eventos similares.
index.add(embeddings_array)

print("FAISS index creado")
print("Vectores guardados:", index.ntotal)

FAISS index creado
Vectores guardados: 2500


In [25]:
faiss.write_index(
    index,
    os.path.join(DATA_PATH, "faiss_index.bin")
)

print("FAISS guardado")

FAISS guardado
